In [6]:
# imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [7]:
# loading trained MLP
class SimpleMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))

    def forward(self, x):

        return self.network(x)

In [8]:
model = SimpleMLP()

state_dict = torch.load(
    "mlp_model.pth", map_location=torch.device("cpu"), weights_only=True
)

model.load_state_dict(state_dict)

model.eval()

print("MLP Loaded")

MLP Loaded


In [9]:
# loading scaler
scaler = joblib.load("mlp_scaler.pkl")

print("Scaler Loaded")

Scaler Loaded


In [10]:
# creating dataset to feed to MLP
df = pd.read_csv("clean_testdata.csv")

mlp_df = df[["quality_score", "best_similarity", "margin", "label"]]

mlp_df.to_csv("clean_mlp.csv", index=False)

print(mlp_df.shape)

mlp_df.head()

(135, 4)


,quality_score,best_similarity,margin,label
0,20.942333,0.548114,0.315820,1
1,23.284578,0.734537,0.501310,1
2,20.365555,0.745469,0.530061,1
3,17.202911,0.486810,0.282815,1
4,17.514805,0.607458,0.355503,1


In [11]:
# running MLP
df = pd.read_csv("clean_mlp.csv")

X = df[["quality_score", "best_similarity", "margin"]]

y_true = df["label"]

X_scaled = scaler.transform(X)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

with torch.no_grad():

    outputs = model(X_tensor)

    probabilities = torch.softmax(outputs, dim=1)

    y_pred = torch.argmax(outputs, dim=1).numpy()

df["mlp_decision"] = y_pred

df.to_csv("clean_mlp_predictions.csv", index=False)

print("Saved clean_mlp_predictions.csv")

Saved clean_mlp_predictions.csv


In [12]:
# MLP metrics
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, zero_division=0)

recall = recall_score(y_true, y_pred, zero_division=0)

f1 = f1_score(y_true, y_pred, zero_division=0)

print("===== MLP RESULTS =====\n")

print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)

cm_mlp = confusion_matrix(y_true, y_pred)

print("\nConfusion Matrix")

print(cm_mlp)

print(classification_report(y_true, y_pred, zero_division=0))

===== MLP RESULTS =====

Accuracy : 0.9407407407407408
Precision: 1.0
Recall   : 0.9393939393939394
F1 Score : 0.96875

Confusion Matrix
[[  3   0]
 [  8 124]]
              precision    recall  f1-score   support

           0       0.27      1.00      0.43         3
           1       1.00      0.94      0.97       132

    accuracy                           0.94       135
   macro avg       0.64      0.97      0.70       135
weighted avg       0.98      0.94      0.96       135



In [13]:
# TAR TRR FAR FRR
TN, FP, FN, TP = cm_mlp.ravel()

TAR = TP / (TP + FN)

FRR = FN / (TP + FN)

TRR = TN / (TN + FP)

FAR = FP / (TN + FP)

print("\n===== VERIFICATION METRICS =====")

print("TAR:", TAR)

print("FRR:", FRR)

print("TRR:", TRR)

print("FAR:", FAR)


===== VERIFICATION METRICS =====
TAR: 0.9393939393939394
FRR: 0.06060606060606061
TRR: 1.0
FAR: 0.0


In [14]:
# fixed threshold
threshold = 0.6

threshold_pred = (df["best_similarity"] >= threshold).astype(int)

accuracy_th = accuracy_score(y_true, threshold_pred)

precision_th = precision_score(y_true, threshold_pred, zero_division=0)

recall_th = recall_score(y_true, threshold_pred, zero_division=0)

f1_th = f1_score(y_true, threshold_pred, zero_division=0)

print("===== THRESHOLD RESULTS =====\n")

print("Accuracy :", accuracy_th)

print("Precision:", precision_th)

print("Recall   :", recall_th)

print("F1 Score :", f1_th)

cm_threshold = confusion_matrix(y_true, threshold_pred)

print("\nConfusion Matrix")

print(cm_threshold)

print(classification_report(y_true, threshold_pred, zero_division=0))

TN, FP, FN, TP = cm_threshold.ravel()

TAR_th = TP / (TP + FN)

FRR_th = FN / (TP + FN)

TRR_th = TN / (TN + FP)

FAR_th = FP / (TN + FP)

print("\n===== THRESHOLD METRICS =====")

print("TAR:", TAR_th)

print("FRR:", FRR_th)

print("TRR:", TRR_th)

print("FAR:", FAR_th)

===== THRESHOLD RESULTS =====

Accuracy : 0.7333333333333333
Precision: 1.0
Recall   : 0.7272727272727273
F1 Score : 0.8421052631578947

Confusion Matrix
[[ 3  0]
 [36 96]]
              precision    recall  f1-score   support

           0       0.08      1.00      0.14         3
           1       1.00      0.73      0.84       132

    accuracy                           0.73       135
   macro avg       0.54      0.86      0.49       135
weighted avg       0.98      0.73      0.83       135


===== THRESHOLD METRICS =====
TAR: 0.7272727272727273
FRR: 0.2727272727272727
TRR: 1.0
FAR: 0.0


In [15]:
# final comparison table
svm_accuracy = 0.9703703703703703

svm_precision = 0.99

svm_recall = 0.9773

svm_f1 = 0.98

svm_tar = 0.977273

svm_frr = 0.022727

svm_trr = 0.666667

svm_far = 0.333333

comparison = pd.DataFrame(
    {
        "Method": ["Threshold", "SVM", "MLP"],
        "Accuracy": [accuracy_th, svm_accuracy, accuracy],
        "Precision": [precision_th, svm_precision, precision],
        "Recall": [recall_th, svm_recall, recall],
        "F1": [f1_th, svm_f1, f1],
        "TAR": [TAR_th, svm_tar, TAR],
        "FRR": [FRR_th, svm_frr, FRR],
        "TRR": [TRR_th, svm_trr, TRR],
        "FAR": [FAR_th, svm_far, FAR],
    }
)

print("\n===== FINAL COMPARISON =====\n")

comparison.round(4)


===== FINAL COMPARISON =====



,Method,Accuracy,Precision,Recall,F1,TAR,FRR,TRR,FAR
0,Threshold,0.7333,1.00,0.7273,0.8421,0.7273,0.2727,1.0000,0.0000
1,SVM,0.9704,0.99,0.9773,0.9800,0.9773,0.0227,0.6667,0.3333
2,MLP,0.9407,1.00,0.9394,0.9688,0.9394,0.0606,1.0000,0.0000
